In [1]:
from pyspark.sql import SparkSession

# Khởi tạo Spark Session
spark = SparkSession.builder.appName("RetailAnalysis").getOrCreate()

# 1. Đọc file CSV 
df = spark.read.csv("data/retail_transactions.csv", header=True, inferSchema=True)

# 2. Hiển thị 10 dòng đầu tiên
print("Dữ liệu 10 dòng đầu tiên")
df.show(10)

# 3. In cấu trúc của dữ liệu
print("Cấu trúc Schema")
df.printSchema()

Dữ liệu 10 dòng đầu tiên
+--------------+--------+----------+--------+--------+-----+----------------+
|transaction_id|store_id|product_id|category|quantity|price|transaction_date|
+--------------+--------+----------+--------+--------+-----+----------------+
|             1|     S01|      P100|   Drink|       2| 15.5|      2026-04-01|
|             2|     S01|      P101|   Snack|       3| 12.0|      2026-04-01|
|             3|     S02|      P100|   Drink|       1| 15.5|      2026-04-01|
|             4|     S03|      P103|Cleaning|       5|  8.0|      2026-04-01|
+--------------+--------+----------+--------+--------+-----+----------------+

Cấu trúc Schema
root
 |-- transaction_id: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- transaction_date: date (nullable = true)



In [ ]:
# Phần B làm sạch dữ liệu

In [2]:
from pyspark.sql.functions import col, sum, desc
df_clean = df.dropna(subset=["store_id", "product_id", "quantity", "price"])
df_clean = df_clean.filter((col("quantity") > 0) & (col("price") > 0))
print("Dữ liệu sau khi làm sạch (5 dòng đầu):")
df_clean.show(5)

Dữ liệu sau khi làm sạch (5 dòng đầu):
+--------------+--------+----------+--------+--------+-----+----------------+
|transaction_id|store_id|product_id|category|quantity|price|transaction_date|
+--------------+--------+----------+--------+--------+-----+----------------+
|             1|     S01|      P100|   Drink|       2| 15.5|      2026-04-01|
|             2|     S01|      P101|   Snack|       3| 12.0|      2026-04-01|
|             3|     S02|      P100|   Drink|       1| 15.5|      2026-04-01|
|             4|     S03|      P103|Cleaning|       5|  8.0|      2026-04-01|
+--------------+--------+----------+--------+--------+-----+----------------+



In [ ]:
# Phần C tạo biến mới

In [3]:
# Tạo cột mới revenue = quantity * price
df_enriched = df_clean.withColumn("revenue", col("quantity") * col("price"))

print("Bảng dữ liệu sau khi thêm cột Doanh thu (revenue):")
df_enriched.show(5)

Bảng dữ liệu sau khi thêm cột Doanh thu (revenue):
+--------------+--------+----------+--------+--------+-----+----------------+-------+
|transaction_id|store_id|product_id|category|quantity|price|transaction_date|revenue|
+--------------+--------+----------+--------+--------+-----+----------------+-------+
|             1|     S01|      P100|   Drink|       2| 15.5|      2026-04-01|   31.0|
|             2|     S01|      P101|   Snack|       3| 12.0|      2026-04-01|   36.0|
|             3|     S02|      P100|   Drink|       1| 15.5|      2026-04-01|   15.5|
|             4|     S03|      P103|Cleaning|       5|  8.0|      2026-04-01|   40.0|
+--------------+--------+----------+--------+--------+-----+----------------+-------+



In [ ]:
# Phần D phân tích

In [5]:
#  Tính tổng doanh thu toàn bộ hệ thống
total_revenue = df_enriched.select(sum("revenue")).collect()[0][0]
print(f". Tổng doanh thu toàn hệ thống: {total_revenue}")

# Tính doanh thu theo từng cửa hàng (store_id)
print("\n.Doanh thu theo từng cửa hàng:")
df_enriched.groupBy("store_id").agg(sum("revenue").alias("total_revenue")).show()

# Tính doanh thu theo từng danh mục sản phẩm (category)
print("Doanh thu theo từng danh mục sản phẩm:")
df_enriched.groupBy("category").agg(sum("revenue").alias("total_revenue")).show()

# Tìm 10 sản phẩm có doanh thu cao nhất (product_id)
print("Top 10 sản phẩm có doanh thu cao nhất:")
df_enriched.groupBy("product_id") \
    .agg(sum("revenue").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .show(10)

. Tổng doanh thu toàn hệ thống: 122.5

.Doanh thu theo từng cửa hàng:
+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|     S02|         15.5|
|     S01|         67.0|
|     S03|         40.0|
+--------+-------------+

Doanh thu theo từng danh mục sản phẩm:
+--------+-------------+
|category|total_revenue|
+--------+-------------+
|   Drink|         46.5|
|   Snack|         36.0|
|Cleaning|         40.0|
+--------+-------------+

Top 10 sản phẩm có doanh thu cao nhất:
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|      P100|         46.5|
|      P103|         40.0|
|      P101|         36.0|
+----------+-------------+



Câu 1: Vì sao PySpark phù hợp hơn pandas khi dữ liệu rất lớn?
Pandas: Được thiết kế để chạy trên một máy tính duy nhất (single-node) và nạp toàn bộ dữ liệu vào bộ nhớ RAM (in-memory). Nếu file dữ liệu vượt quá dung lượng RAM của máy, chương trình sẽ báo lỗi (Out of Memory) hoặc chạy cực kỳ chậm.

PySpark: Sử dụng cơ chế xử lý tính toán phân tán (distributed computing). Thay vì nạp tất cả vào một máy, PySpark chia nhỏ khối lượng dữ liệu khổng lồ ra thành nhiều phần (partitions) và xử lý song song trên nhiều lõi CPU hoặc nhiều máy tính khác nhau trong một cụm (cluster). Do đó, nó có thể dễ dàng xử lý hàng Terabyte dữ liệu mà không bị giới hạn bởi RAM của một máy duy nhất.

Câu 2: Trong bài toán này, đâu là ví dụ của "descriptive analytics"?
Descriptive Analytics (Phân tích mô tả) là bước phân tích dữ liệu nhằm trả lời cho câu hỏi: "Chuyện gì đã xảy ra trong quá khứ?" bằng cách tổng hợp và tóm tắt dữ liệu lịch sử.

Trong bài tập này, toàn bộ Phần D chính là ví dụ điển hình của Descriptive Analytics. Các thao tác như: Tính tổng doanh thu toàn hệ thống, gom nhóm tính doanh thu theo từng cửa hàng (store_id) / danh mục (category), và tìm Top 10 sản phẩm bán chạy nhất đều là việc chúng ta đang "mô tả" lại bức tranh kinh doanh từ tập dữ liệu giao dịch đã diễn ra.

Câu 3: Nếu dữ liệu tăng lên 100 triệu giao dịch, lợi thế của Spark là gì?
Nếu dữ liệu phình to lên 100 triệu dòng (vài chục GB), lợi thế của Spark sẽ cực kỳ rõ rệt ở 3 điểm:
Tốc độ xử lý song song: Spark sẽ tự động chia 100 triệu dòng này ra cho nhiều lõi CPU cùng tính toán đồng thời (ví dụ: tính tổng doanh thu cùng lúc ở nhiều khối dữ liệu rồi cộng gộp lại), giúp thời gian xử lý nhanh hơn gấp nhiều lần so với việc đọc tuần tự từng dòng.

Lazy Evaluation (Đánh giá lười biếng): Spark rất thông minh, nó không thực hiện tính toán ngay khi bạn gán biến, mà chỉ lập ra một "kế hoạch" (DAG). Chỉ khi bạn gọi một "Action" Spark mới thực sự bắt tay vào xử lý. Điều này giúp hệ thống tối ưu hóa đường đi và không bị lãng phí tài nguyên vô ích.

Khả năng mở rộng (Scalability): Nếu 100 triệu dòng làm máy hiện tại bị quá tải, bạn chỉ cần cắm thêm các máy tính khác (nodes) vào mạng lưới của Spark (cluster). Hệ thống sẽ tự động san sẻ gánh nặng sang các máy mới mà không cần phải viết lại code.